## Parallel batch reasoning

#### Solve first stage : Biomass collecting regions --> Depôts

In [22]:
import pandas as pd
from pulp import *
from threading import Thread

def solve_batch(batch_num, batch_increment,biomass_data, distance_matrix):

    surplus = 0
    message = ""

    if batch_num == ((2417//batch_increment)-1)*batch_increment:
        surplus = 18

    # # Step 2: Read the provided datasets
    biomass_data = biomass_data.iloc[batch_num:batch_num+batch_increment+surplus,:4]

    distance_matrix = distance_matrix.iloc[batch_num:batch_num+batch_increment+surplus,1+batch_num:1+batch_num+batch_increment+surplus]
       
    # Step 3: Create PuLP variables and the optimization problem
    model =  LpProblem("Waste_to_Energy_Optimization",LpMinimize)

    ####################################### Parameters ###############################################
    num_harvesting_sites = len(biomass_data)
    
    num_depots = 25

    ################################### Decision variables ##########################################

    # Amount of biomass transported from each Harvesting Site to each Depot (continuous variable)
    biomass_demand_supply = LpVariable.dicts("biomass_demand_supply",[(i, j) for i in range(num_harvesting_sites) \
                                                        for j in range(num_harvesting_sites)], lowBound=0, cat=LpContinuous) 
    
    # Binary variables representing whether Depot j and Biorefinery k are placed or not
    depot = LpVariable.dicts("depot_location",range(num_harvesting_sites), cat=LpBinary)

    #################################### Objective function ############################################

    # Constants
    a = 0.001
    c = 1
    cap_depot = 20000


    cost_transport = lpSum(distance_matrix.iloc[i, j] * biomass_demand_supply[i, j] for i in range(num_harvesting_sites) \
                                                                                    for j in range(num_harvesting_sites))
    cost_underutilization = lpSum((depot[j]*cap_depot - lpSum(biomass_demand_supply[i, j] for i in range(num_harvesting_sites))) \
                                                                                          for j in range(num_harvesting_sites)) 

    # Objective function
    model += a * cost_transport + c * cost_underutilization, "Total_Cost"

    ####################################### Constraints ###############################################
   
    # Constraint: Biomass demand from each Harvesting Site i must be less than or equal to its forecasted biomass availability
    for i in range(num_harvesting_sites):
        model += lpSum(biomass_demand_supply[i, j] for j in range(num_harvesting_sites)) <= biomass_data[f'{biomass_data.columns[3]}'].iloc[i]

    # Constraint: Total biomass reaching each preprocessing depot j must be less than or equal to its yearly processing capacity (20,000)
    for j in range(num_harvesting_sites):
        model += lpSum(biomass_demand_supply[i, j] for i in range(num_harvesting_sites)) <= depot[j]*cap_depot

    # Constraint: Limit the number of depots to be less than or equal to 25
    model += lpSum(depot[j] for j in range(num_harvesting_sites)) <= num_depots


    # Constraint: At least 80% of the total forecasted biomass is processed by biorefineries each year
    total_forecasted_biomass = sum(biomass_data[f'{biomass_data.columns[3]}'])
    total_processed_biomass = lpSum(biomass_demand_supply[i, j] for i in range(num_harvesting_sites) for j in range(num_harvesting_sites))
    model += total_processed_biomass >= 0.8 * total_forecasted_biomass

    #################################### Solving the problem ############################################

    solver = CPLEX_CMD()
    model.solve(solver=solver)
    message += f"Status {batch_num}: {LpStatus[model.status]}\n"

    ####################################### Reporting ###############################################

    for v in model.variables():
        if v.varValue!=0 and v.varValue!=None:
            if 'biomass' in v.name:
                splitted_name = (v.name).split("_")
                message += f"2018,{'_'.join(splitted_name[0:-2])},{batch_num+int(splitted_name[-2].lstrip('(').rstrip(','))},{batch_num+int(splitted_name[-1].rstrip(')'))},{v.varValue}\n"
            else:
                splitted_name = (v.name).split("_")
                message += f"2018,{'_'.join(splitted_name[0:2])},{batch_num+int(splitted_name[-1])},,\n"

    # Calculate the sum of values for each decision variable
    biomass_sum = lpSum(biomass_demand_supply[i, j].varValue for i in range(num_harvesting_sites) for j in range(num_harvesting_sites))
    depot_sum = lpSum(depot[j].varValue for j in range(num_harvesting_sites))

    message += f"Total biomass harvested: {biomass_sum}\n"
    message += f"Number of depots: {depot_sum}\n"

    with open(f'Report_batch_{batch_num}_to_{batch_num+batch_increment+surplus-1}.txt','w') as file:
        file.write(message)

# Load the data outside the threads
biomass_data = pd.read_csv('Biomass_History.csv')
distance_matrix = pd.read_csv('hackerank/dataset/Distance_Matrix.csv')
batch_increment = 200

threads = []
for i in range(0, (400//batch_increment - 1) * batch_increment + 1, batch_increment):
    # Create separate copies of data for each thread
    batch_biomass_data = biomass_data.copy()
    batch_distance_matrix = distance_matrix.copy()

    t = Thread(target=solve_batch, args=(i, batch_increment,batch_biomass_data, batch_distance_matrix))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

#### Solve second stage : Depôts --> Biorefineries

In [10]:
import pandas as pd
from pulp import *

batch_increment = 2417

def solve_batch(batch_num, biomass_data, distance_matrix):

    surplus = 0
    message = ""

    if batch_num == ((2417//batch_increment)-1)*batch_increment:
        surplus = 18

    # Step 2: Read the provided datasets
    biomass_data = biomass_data.iloc[batch_num:batch_num+batch_increment+surplus,:4]

    distance_matrix = distance_matrix.iloc[batch_num:batch_num+batch_increment+surplus,1+batch_num:1+batch_num+batch_increment+surplus]
       
    # Step 3: Create PuLP variables and the optimization problem
    model =  LpProblem("Waste_to_Energy_Optimization",LpMinimize)

    ####################################### Parameters ###############################################

    num_harvesting_sites = len(biomass_data)
    num_biorefineries = 5

    indices_depots = list(pd.read_csv('collected_biomass.csv')['depots'])
    total_collect_biomass = pd.read_csv('collected_biomass.csv')

    #################################### Decision variables ############################################

    # Amount of biomass transported from each Depot to each refinery (continuous variable)
    pellet_demand_supply = LpVariable.dicts("pellet_demand_supply",[(j, k) for j in range(num_harvesting_sites) \
                                                       for k in range(num_harvesting_sites)], lowBound=0, cat=LpContinuous)
    
    # Binary variables representing whether Depot j and Biorefinery k are placed or not
    # depot = LpVariable.dicts("depot",indices_depots, cat=LpBinary)
    refinery = LpVariable.dicts("refinery_location",[k for k in range(num_harvesting_sites)], cat=LpBinary)

    ################################# The objective function #########################################

    # Constants
    a = 0.001
    c = 1
    cap_refinery = 100000

    # Objective function
    cost_transport = lpSum(distance_matrix.iloc[j, k] * pellet_demand_supply[j, k] for j in indices_depots\
                                                                                    for k in range(num_harvesting_sites))

    cost_underutilization = lpSum((refinery[k]*cap_refinery - lpSum(pellet_demand_supply[j, k] for j in indices_depots)) \
                                                                                          for k in range(num_harvesting_sites))

    # Objective function
    model += a * cost_transport + c * cost_underutilization, "Total_Cost"

    
    ####################################### Constraints ###############################################

    # Constraint: Total pellets reaching each biorefinery k must be less than or equal to its yearly processing capacity (100,000)
    for k in range(num_harvesting_sites):
        model += lpSum(pellet_demand_supply[j, k] for j in indices_depots) <= refinery[k]*cap_refinery

    # Constraint: Limit the number of biorefineries to be less than or equal to 5
    model += lpSum(refinery[k] for k in range(num_harvesting_sites)) <= num_biorefineries


    # Constraint: Total biomass entering each preprocessing depot should be equal to the total amount of pellets exiting that depot (within tolerance limit)
    for j in indices_depots:
        model += total_collect_biomass.loc[total_collect_biomass['depots']==j]['quantities'].item()-\
                lpSum(pellet_demand_supply[j, k] for k in range(num_harvesting_sites)) == 0

    #################################### Solving the problem ############################################

    solver = CPLEX_CMD()
    model.solve(solver=solver)
    message += f"Status {batch_num}: {LpStatus[model.status]}\n"

    for v in model.variables():
        if v.varValue!=0:
            splitted_name = (v.name).split("_")
            if 'location'not in v.name:
                message += f"2018,{'_'.join(splitted_name[0:-2])},{batch_num+int(splitted_name[-2].lstrip('(').rstrip(','))},{batch_num+int(splitted_name[-1].rstrip(')'))},{v.varValue}\n"
            else:
                # !!! Afficher plutôt la quantité
                message += f"2018,{'_'.join(splitted_name[0:2])},{v.varValue}\n"
    ####################################### Reporting ###############################################
  
    # Calculate the sum of values for each decision variable
    pellet_sum = lpSum(pellet_demand_supply[j, k].varValue for j in range(num_harvesting_sites) for k in range(num_harvesting_sites))
    refinery_sum = lpSum(refinery[k].varValue for k in range(num_harvesting_sites))

    message += f"Total pellets processed: {pellet_sum}\n"
    message += f"Nomber of refineries : {refinery_sum}\n"

    with open(f'(2)_Report_batch_{batch_num}_to_{batch_num+batch_increment+surplus-1}.txt','w') as file:
        file.write(message)

# Load the data outside the threads
biomass_data = pd.read_csv('Biomass_History.csv')
distance_matrix = pd.read_csv('hackerank/dataset/Distance_Matrix.csv')

solve_batch(0,biomass_data,distance_matrix)

In [28]:
# import pandas as pd
# biomass_data1 = pd.read_csv('Biomass_History.csv')
# biomass_data1.iloc[:,1:5].set_index("Index").to_csv('Biomass_History.csv')


################################# Formatting forecasting for submission ##############################

# import pandas as pd
# biomass_data1 = pd.read_csv('Biomass_History.csv',index_col="Index")

# biomass_data1 = pd.DataFrame({
#     "year":[2019 for _ in range(2418)],
#     "data_type":["biomass_forecast" for _ in range(2418)],
#     "source_index":list(range(2418)),
#     "value":biomass_data1["2018"]
# })

# biomass_data1["destination_index"] = ""

# biomass_data1 = biomass_data1[["year","data_type","source_index","destination_index","value"]].set_index("year")

# biomass_data1.to_csv("submission_forecast_2019.csv")